# 02 — Metastability and Brownian dynamics

**Learning goals**

- characterize an asymmetric double well: minima, barrier, barrier heights in $k_BT$,
- run overdamped Langevin (Brownian) dynamics and see finite-time trapping,
- understand why a histogram from a trapped trajectory is not an equilibrium free energy,
- know when $F(x) = V(x)$ holds (and when it does not).

This model (System C) is used in all enhanced-sampling notebooks that follow.

In [ ]:
# Setup: make the `src` package importable (locally or on Google Colab).
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/HyeonjeYang/interactive-metadynamics.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/interactive-metadynamics")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
else:
    ROOT = Path.cwd().resolve()
    if not (ROOT / "src").is_dir():
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import inspect
import numpy as np
import matplotlib.pyplot as plt

from src.potentials import (double_well_potential, double_well_gradient, double_well_force,
                            double_well_landmarks, true_free_energy, numerical_derivative)
from src.langevin import brownian_dynamics
from src.analysis import histogram_free_energy, count_transitions
from src.plotting import set_style, save_figure, plot_reference, free_energy_axes, PALETTE, REFERENCE, MUTED
set_style()

## Parameters

In [ ]:
a, b = 2.0, 0.5            # V(x) = a (x^2 - 1)^2 + b x
T = 0.4                    # k_B = 1, so kT = T
friction = 1.0             # zeta
dt = 0.002
n_steps = 100_000          # 200 time units per walker
n_walkers = 8              # independent walkers, all started in the left well
stride = 10                # store every 10th step
seed = 2026
rng = np.random.default_rng(seed)

## 1. The asymmetric double well

$$V(x) = a(x^2-1)^2 + bx, \qquad \frac{dV}{dx} = 4ax(x^2-1) + b.$$

First we check the analytical derivative against a central finite difference.

In [ ]:
x_plot = np.linspace(-1.8, 1.7, 400)
num = numerical_derivative(lambda x: double_well_potential(x, a, b), x_plot)
print(f"max |analytical - numerical| derivative = {np.max(np.abs(double_well_gradient(x_plot, a, b) - num)):.1e}")

lm = double_well_landmarks(a, b)
print(f"left minimum   x = {lm['x_left']:+.3f}   V = {lm['V_left']:+.3f}")
print(f"barrier top    x = {lm['x_barrier']:+.3f}   V = {lm['V_barrier']:+.3f}")
print(f"right minimum  x = {lm['x_right']:+.3f}   V = {lm['V_right']:+.3f}")
print(f"barrier from left  = {lm['barrier_from_left']:.2f} = {lm['barrier_from_left'] / T:.1f} kT")
print(f"barrier from right = {lm['barrier_from_right']:.2f} = {lm['barrier_from_right'] / T:.1f} kT")
print(f"well asymmetry V_right - V_left = {(lm['V_right'] - lm['V_left']) / T:.1f} kT")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(x_plot, double_well_potential(x_plot, a, b), color=REFERENCE, lw=1.6)
for key, label in [("left", "left minimum"), ("barrier", "barrier"), ("right", "right minimum")]:
    ax.plot(lm[f"x_{key}"], lm[f"V_{key}"], "o", ms=8, color=PALETTE[0], mec="white", mew=1.5)
    ax.annotate(label, (lm[f"x_{key}"], lm[f"V_{key}"]), textcoords="offset points", xytext=(0, 10), ha="center")
ax.set_ylim(-1, 4.5)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$V(x)$")
fig.tight_layout()
plt.show()

## 2. The true free energy of this toy model

Because $x$ is the only coordinate, the equilibrium density is $P(x)\propto e^{-\beta V(x)}$ and

$$F_\mathrm{true}(x) = -k_BT\ln P(x) = V(x) - \min V.$$

This identity is special to a one-coordinate (or fully decoupled) model. For a collective variable $s(\mathbf r)$ of a
many-body system, $F(s) = -k_BT\ln\int\delta(s(\mathbf r)-s)\,e^{-\beta V(\mathbf r)}\,d\mathbf r \neq V(s)$ in general,
because the hidden degrees of freedom are integrated out (entropy matters).

## 3. Overdamped Langevin dynamics

$$dx = -\frac{1}{\zeta}\frac{dU}{dx}\,dt + \sqrt{\frac{2k_BT\,dt}{\zeta}}\;\eta, \qquad \eta\sim\mathcal N(0,1).$$

`friction` is $\zeta$ (the name `gamma` is reserved for the metadynamics bias factor later).

In [ ]:
print(inspect.getsource(brownian_dynamics))

**Timestep check.** Near a minimum the deterministic relaxation time is $\zeta/V''$. The timestep should be much
smaller, and one noise kick should be much smaller than the thermal width of the well, $\sqrt{k_BT/V''}$.

In [ ]:
curv = 12 * a * lm["x_left"] ** 2 - 4 * a
print(f"relaxation time zeta/V'' = {friction / curv:.3f}  vs  dt = {dt}   (ratio {dt * curv / friction:.3f})")
print(f"noise kick {np.sqrt(2 * T * dt / friction):.3f}  vs  thermal width {np.sqrt(T / curv):.3f}")

## 4. Unbiased trajectories from the left well

In [ ]:
traj = brownian_dynamics(np.full(n_walkers, lm["x_left"]), lambda x: double_well_force(x, a, b),
                         n_steps, dt, T, rng, friction, stride=stride)
time = np.arange(len(traj)) * stride * dt

core = (lm["x_left"] + 0.3, lm["x_right"] - 0.3)   # committed-basin cores
crossings = count_transitions(traj, *core)
print("transitions per walker:", crossings)

In [ ]:
show = [int(np.argmax(crossings)), int(np.argmin(crossings))]
fig, axes = plt.subplots(2, 1, figsize=(8, 4.6), sharex=True, gridspec_kw={"height_ratios": [2, 1.2]})
for w, color in zip(show, PALETTE):
    axes[0].plot(time, traj[:, w], lw=0.6, color=color, label=f"walker {w} ({crossings[w]} transitions)")
axes[0].axhline(lm["x_barrier"], color=MUTED, ls=":", lw=1)
axes[0].set_ylabel(r"$x(t)$")
axes[0].legend(loc="upper left", fontsize=8)
axes[1].imshow((traj > lm["x_barrier"]).T, aspect="auto", interpolation="nearest",
               cmap=plt.matplotlib.colors.ListedColormap(["#cde2fb", PALETTE[1]]),
               extent=[0, time[-1], n_walkers - 0.5, -0.5])
axes[1].grid(False)
axes[1].set_ylabel("walker")
axes[1].set_xlabel("time (reduced units)")
axes[1].set_title("light: left well, orange: right of the barrier", fontsize=9)
fig.tight_layout()
save_figure(fig, "02_metastability")
plt.show()

## 5. How long should we expect to wait?

For a high barrier, Kramers' overdamped rate is $k \approx \frac{\sqrt{V''_\mathrm{min}|V''_\mathrm{b}|}}{2\pi\zeta}e^{-\beta\Delta V}$.
It is an approximation, used here only to set expectations.

In [ ]:
def kramers_time(x_min):
    w_min = 12 * a * x_min**2 - 4 * a
    w_bar = abs(12 * a * lm["x_barrier"] ** 2 - 4 * a)
    rate = np.sqrt(w_min * w_bar) / (2 * np.pi * friction) * np.exp(-(lm["V_barrier"] - double_well_potential(x_min, a, b)) / T)
    return 1.0 / rate

print(f"Kramers mean escape time from the left well  ~ {kramers_time(lm['x_left']):.0f}")
print(f"Kramers mean escape time from the right well ~ {kramers_time(lm['x_right']):.0f}")
print(f"simulated time per walker = {time[-1]:.0f}")

Each walker runs for less than one typical escape time from the deep well, so some walkers never leave and others
escape once or twice. This is *finite-time* metastability: nothing forbids the transition, it is just rare on the
simulated timescale.

## 6. What does the sampled histogram tell us?

In [ ]:
bins = np.linspace(-1.8, 1.7, 71)
xc, F_est, counts = histogram_free_energy(traj[1:], bins, T, min_count=5)
F_ref = true_free_energy(xc, a, b)

p_right_sampled = np.mean(traj[1:] > lm["x_barrier"])
w = np.exp(-F_ref / T)
p_right_exact = w[xc > lm["x_barrier"]].sum() / w.sum()
print(f"fraction of time right of the barrier: sampled {p_right_sampled:.3f}, equilibrium {p_right_exact:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].bar(xc, counts / counts.sum() / np.diff(bins), width=np.diff(bins), color=PALETTE[0], alpha=0.6,
            label="all walkers (pooled)")
axes[0].plot(xc, w / w.sum() / np.diff(bins), ls="--", color=REFERENCE, lw=1.4, label=r"$e^{-\beta V}/Z$")
axes[0].set_yscale("log")
axes[0].set_ylim(1e-4, 5)
axes[0].set_xlabel(r"$x$")
axes[0].set_ylabel("probability density")
axes[0].legend(fontsize=8)
plot_reference(axes[1], xc, F_ref)
axes[1].plot(xc, F_est, "o-", ms=3, color=PALETTE[0], label=r"$-k_BT\ln P_\mathrm{sampled}$ (bins with $\geq$5 samples)")
free_energy_axes(axes[1], ylim=(-0.2, 3.6))
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

The deep left well is sampled well. The right-well probability depends on the handful of transitions that
happened to occur, and the barrier region is nearly empty, so there is no estimate there (bins with fewer than
5 samples are left blank instead of being assigned an arbitrary large value). The *relative* free energy of the
two basins is therefore poorly determined — this is the sampling problem that the next notebooks address.

## 7. Optional experiment

How strongly does the transition count depend on temperature? Set `RUN_OPTIONAL = True`.
Higher temperatures shorten the escape time roughly as $e^{\Delta V/k_BT}$.

In [ ]:
RUN_OPTIONAL = False
if RUN_OPTIONAL:
    for T_test in (0.3, 0.4, 0.6, 0.8):
        tr = brownian_dynamics(np.full(n_walkers, lm["x_left"]), lambda x: double_well_force(x, a, b),
                               n_steps, dt, T_test, np.random.default_rng(seed), friction, stride=stride)
        print(f"T = {T_test}: barrier = {lm['barrier_from_left'] / T_test:4.1f} kT, "
              f"transitions per walker = {count_transitions(tr, *core)}")

## Takeaways

- With a barrier of about $6\,k_BT$, unbiased Brownian dynamics stays in the starting well for long stretches of time.
- Finite-time trapping biases histogram-based free energies, especially the relative depth of the two basins.
- Empty bins carry no information and should not be filled with artificial numbers.
- $F(x) = V(x) - \min V$ holds only because this toy model has a single coordinate.